In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import cohen_kappa_score, roc_auc_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dl = pd.read_csv('data/data_dl.csv')

In [3]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [4]:
num_cols = data_dl.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = data_dl.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

In [5]:
cat_cols.remove('Target')
cat_cols

['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']

In [6]:
# Ordinal encoding — preserve the natural order
label_map = {"Poor": 0, "Standard": 1, "Good": 2}
data_dl["Target_label"] = data_dl["Target"].map(label_map)

In [20]:
data_dl[data_dl.isnull().any(axis=1)]

,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,month3_balance,month3_credit_util,month3_delayed_payment,month3_delay_from_due_date,max_delayed_payment,max_credit_util,credit_util_var,balance_m1_m2,balance_m2_m3,Target_label
3,55,Entrepreneur,30689.89,2612.490833,2,5,4,100,4,9.0,...,419.880784,27.445422,6.0,5,9.0,41.154317,34.213323,-0.145042,0.056371,1
5,31,Lawyer,73928.46,5988.705000,4,5,8,0,8,7.0,...,633.080175,35.275437,7.0,7,10.0,42.769864,32.667415,0.560140,-0.193844,2
8,24,Doctor,114838.41,9843.867500,2,5,7,3,11,11.0,...,491.113388,26.114214,14.0,13,14.0,35.141567,9.717689,-0.061997,0.612714,1
9,45,Journalist,31370.80,NaN,1,6,12,2,-2,2.0,...,378.189779,25.502469,0.0,1,2.0,37.565053,20.705851,-0.190363,-0.107558,2
11,33,Engineer,88640.24,7266.686667,3,6,1,2,-1,NaN,...,195.369079,38.323639,0.0,4,2.0,41.036168,25.202502,0.098338,2.416203,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12490,38,Media_Manager,139664.96,11777.746667,3,6,12,4,14,12.0,...,182.547539,27.714375,12.0,14,12.0,40.171095,22.103331,NaN,2.482677,2
12492,48,_______,22620.79,NaN,6,2,9,0,27,15.0,...,254.733128,27.699504,18.0,27,19.0,37.450793,18.993999,-0.350503,0.553290,0
12495,19,Lawyer,42903.79,3468.315833,0,4,6,1,9,NaN,...,394.500158,35.549456,-1.0,14,0.0,35.716618,19.951316,0.272631,-0.110311,2
12496,45,Media_Manager,16680.35,NaN,1,1,5,4,1,0.0,...,233.301539,24.972853,NaN,1,0.0,41.212367,27.790639,-0.087474,0.460277,2


In [7]:
target_cols = ['Target_label']

In [8]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight= 'balanced',
    classes=np.array([0, 1, 2]),
    y=data_dl["Target_label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
# Pass into loss function — model pays more attention to rare classes
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [9]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [21]:
def null_fix(df):
    for col in df.columns:
        if col == "Monthly_Inhand_Salary" and df[col].isnull().any():
            df[col] = df['Annual_Income'] / 12
        elif df[col].isnull().any():
            df[col].fillna(0, inplace=True)
    return df

In [22]:
data_dl = data_dl.pipe(null_fix)

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_39468/904450771.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)


In [23]:
X_train, X_test , y_train , y_test = train_test_split(
    data_dl.drop(columns=target_cols),
    data_dl[target_cols],
    test_size=0.25,
    random_state=42,
    stratify=data_dl[target_cols]
)

In [24]:
print("Checking for NaN in training data...")
print(f"NaN in X_train: {X_train.isnull().sum().sum()}")
print(f"NaN in X_test: {X_test.isnull().sum().sum()}")
if X_train.isnull().any().any():
    print("\nColumns with NaN in train set:")
    print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
    # Fill NaN with median for numeric, mode for categorical
    for col in num_cols:
        if X_train[col].isnull().any():
            X_train[col].fillna(X_train[col].median(), inplace=True)
            X_test[col].fillna(X_test[col].median(), inplace=True)
    for col in cat_cols:
        if X_train[col].isnull().any():
            X_train[col].fillna(X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else "Unknown", inplace=True)
            X_test[col].fillna(X_test[col].mode()[0] if len(X_test[col].mode()) > 0 else "Unknown", inplace=True)
    print("NaN values filled.")

Checking for NaN in training data...
NaN in X_train: 0
NaN in X_test: 0


In [25]:
X_train.head()

,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,month2_delay_from_due_date,month3_balance,month3_credit_util,month3_delayed_payment,month3_delay_from_due_date,max_delayed_payment,max_credit_util,credit_util_var,balance_m1_m2,balance_m2_m3
3738,40,Musician,20877.19,1739.765833,7,6,20,5,20,20.0,...,20,292.372613,38.528879,17.0,17,21.0,38.528879,20.849362,-0.227512,0.040113
7274,40,Accountant,60015.00,5001.250000,7,7,24,2,14,0.0,...,14,542.316098,27.364181,12.0,14,18.0,39.142384,32.646463,0.555936,-0.299054
4810,29,Writer,19220.61,1601.717500,6,7,30,100,43,22.0,...,43,202.294673,33.173982,22.0,47,24.0,40.270504,30.038507,-0.123344,0.502679
121,37,Journalist,18717.02,1559.751667,8,5,15,5,4,18.0,...,7,274.900823,31.583289,16.0,9,20.0,33.813130,8.082557,0.056698,0.005579
603,33,Architect,122942.28,10245.190000,2,3,3,1,8,4.0,...,8,831.347622,32.334353,4.0,8,5.0,41.735281,17.316611,-0.638297,0.254215


In [26]:
label_encoder = LabelEncoder().fit(data_dl[cat_cols].values.ravel())
scaler = StandardScaler().fit(X_train[num_cols])

In [27]:
y = y_train['Target_label'].astype('category')

In [45]:
# Train one encoder per categorical column
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(data_dl[col].astype(str))         # fit on full data
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    le_dict[col] = le

# Now calculate cat_dims correctly
cat_dims = []
for col in cat_cols:
    vocab_size = len(le_dict[col].classes_) + 2 
    embedding_dim = min(8, (vocab_size + 1) // 2)
    cat_dims.append((vocab_size, embedding_dim))

print("cat_dims:", cat_dims)

cat_dims: [(18, 8), (6, 3), (5, 3), (9, 5)]


In [47]:
le_dict

{'Occupation': LabelEncoder(),
 'Credit_Mix': LabelEncoder(),
 'Payment_of_Min_Amount': LabelEncoder(),
 'Payment_Behaviour': LabelEncoder()}

In [37]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
# X_test[num_cols]  = scaler.transform(X_test[num_cols])

In [48]:
class HistDataset(Dataset):
    def __init__(self, X, y, cat_cols, num_cols):
        # X already has integer-encoded cat columns — no re-encoding needed
        self.cat = torch.tensor(X[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(X[num_cols].values, dtype=torch.float32)
        y_vals = y.values.ravel() if hasattr(y, 'values') else np.array(y).ravel()
        self.y  = torch.tensor(y_vals, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat[idx], self.num[idx], self.y[idx]

In [49]:
# print(X_train[cat_cols].head()) 

In [50]:
train_set = HistDataset(X_train, y_train, cat_cols, num_cols)
val_set = HistDataset(X_test, y_test, cat_cols, num_cols)

In [51]:
train_loader = DataLoader(train_set, batch_size= 512, shuffle = True)
val_loader = DataLoader(val_set, batch_size= 512, shuffle = False)

In [52]:
# cat_dims = []
# for col in cat_cols:
#     # Get unique values from the full dataset before split
#     vocab_size = len(data_dl[col].unique()) + 1  # +1 for safety
#     embedding_dim = min(50, (vocab_size + 1) // 2)
#     cat_dims.append((vocab_size, embedding_dim))

# print("cat_dims:", cat_dims)

In [53]:
# num_cols = [c for c in base_features if c not in mlp_cat_cols]


### MLP Classifier

In [54]:
class MLPClassifier(nn.Module):
    def __init__(self, cat_dims, num_cols, hidden = [128, 64, 32], dropout = 0.3):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(v, d) for v, d in cat_dims])
        in_d = sum(d for _, d in cat_dims) + len(num_cols)
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers.append(nn.Linear(in_d, 3))
        self.net = nn.Sequential(*layers)
        
    def forward(self, cx, nx):
        x = torch.cat([e(cx[:, i]) for i, e in enumerate(self.embs)] + [nx], dim=1)
        return self.net(x)

In [55]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [56]:
mlp_clf = MLPClassifier(cat_dims=cat_dims, num_cols=num_cols, hidden=[32, 16], dropout=0.3)
mlp_clf.to(device)
print(f"Parameters : {sum(p.numel() for p in mlp_clf.parameters()):,}")
print(f"Cat embeds : {list(zip(cat_cols, [d for _, d in cat_dims]))}")

Parameters : 2,817
Cat embeds : [('Occupation', 8), ('Credit_Mix', 3), ('Payment_of_Min_Amount', 3), ('Payment_Behaviour', 5)]


In [57]:
class_weight_dl = torch.tensor(class_weights, dtype = torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight = class_weight_dl)

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_39468/2186993042.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  class_weight_dl = torch.tensor(class_weights, dtype = torch.float32).to(device)


In [85]:
optimizer = torch.optim.AdamW(mlp_clf.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

In [86]:
# # Check training data
# print("Train set - cat has NaN:", torch.isnan(torch.tensor(X_train[cat_cols].values)).any())
# print("Train set - num has NaN:", torch.isnan(torch.tensor(X_train[num_cols].values)).any())
# print("Train set - labels have NaN:", torch.isnan(torch.tensor(y_train.values)).any())

# # Check first batch
# cb, num_b, yb = next(iter(train_loader))
# print("First batch - cat has NaN:", torch.isnan(cb.float()).any())
# print("First batch - num has NaN:", torch.isnan(num_b.float()).any())
# print("First batch - labels have NaN:", torch.isnan(yb.float()).any())

In [89]:
def train_dl(model, train_loader, val_loader, criterion, optimizer, scheduler, device,
             max_epochs=300, patience=50):
    best_val_auc, best_state = 0.0, None
    patience_count = 0
    history = {'train_loss': [], 'val_auc': []}

    for epoch in range(max_epochs):
        model.train()
        ep_loss = 0.0
        nan_loss = False
        for cb, num_b, yb in train_loader:
            cb, num_b, yb = cb.to(device), num_b.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(cb, num_b), yb)
            loss.backward()
            # Inside training loop, after loss.backward():
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            ep_loss += loss.item()
            
            # Check for NaN loss
            if torch.isnan(loss):
                print("NaN loss detected! Stopping.")
                nan_loss = True
                break
        if nan_loss:
            break
        mlp_clf.eval()
        vp, vt = [], []; nan_found = False
        with torch.no_grad():
            for cb, num_b, yb in val_loader:
                logits = mlp_clf(cb.to(device), num_b.to(device))
                pred = torch.softmax(logits, dim=1).cpu()
                
                if torch.isnan(pred).any():
                    print(f"NaN in predictions at epoch {epoch}")
                    nan_found = True
                    break
                
                vp.append(pred)
                vt.append(yb)

        if nan_found or len(vp) == 0:
            print("Skipping AUC computation — NaN predictions.")
            continue   # skip to next epoch

        vp_np = torch.cat(vp).numpy()
        vt_np = torch.cat(vt).numpy()

        # For multiclass AUC, use the probability of the true class
        val_auc = roc_auc_score(vt_np, vp_np, multi_class='ovr')  # or 'ovo'
        # val_auc = roc_auc_score(vt_np, vp_np)
        scheduler.step(val_auc)

        history['train_loss'].append(ep_loss / len(train_loader))
        history['val_auc'].append(val_auc)

        if val_auc > best_val_auc:
            best_val_auc, patience_count = val_auc, 0
            best_state = {k: v.clone() for k, v in mlp_clf.state_dict().items()}
        else:
            patience_count += 1

        if patience_count >= 50:
            print(f"Early stopping at epoch {epoch + 1}  |  Best Val AUC: {best_val_auc:.4f}")
            break
        if (epoch + 1) % 30 == 0:
            print(f"Epoch {epoch+1:3d}  loss: {ep_loss/len(train_loader):.4f}  val AUC: {val_auc:.4f}")

    mlp_clf.load_state_dict(best_state)
    # gc.collect()
    return history

In [90]:
mlp_train_result = train_dl(mlp_clf, train_loader, val_loader, criterion, optimizer, scheduler, device,
         max_epochs=300, patience=50)

Epoch  30  loss: 1.0005  val AUC: 0.4999
Early stopping at epoch 54  |  Best Val AUC: 0.5002


In [94]:
class LSTMClassifier(nn.Module):
    def __init__(self, cat_dims, num_cols, hidden = [128, 64, 32], dropout = 0.3):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size =sum(d for _, d in cat_dims) + len(num_cols), 
            hidden_size=hidden[0], batch_first=True)
        self.embs = nn.ModuleList([nn.Embedding(v, d) for v, d in cat_dims])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden[0], hidden[1])
        self.net = nn.Sequential(
            self.dropout,
            self.fc,
            nn.BatchNorm1d(hidden[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[1], hidden[2]),
            nn.BatchNorm1d(hidden[2]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[2], 3)
        )
        
        
    def forward(self, cx, nx):
        x = torch.cat([e(cx[:, i]) for i, e in enumerate(self.embs)] + [nx], dim=1)
        x = x.unsqueeze(1)  # Add sequence dimension
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Take last time step
        return self.net(lstm_out)

In [96]:
lstm_clf = LSTMClassifier(cat_dims=cat_dims, num_cols=num_cols, hidden=[128, 64, 32], dropout=0.3)
lstm_clf.to(device)
print(f"Parameters : {sum(p.numel() for p in lstm_clf.parameters()):,}")
print(f"Cat embeds : {list(zip(cat_cols, [d for _, d in cat_dims]))}")

Parameters : 107,617
Cat embeds : [('Occupation', 8), ('Credit_Mix', 3), ('Payment_of_Min_Amount', 3), ('Payment_Behaviour', 5)]


In [97]:
lstm_train_result = train_dl(lstm_clf, train_loader, val_loader, criterion, optimizer, scheduler, device,
         max_epochs=300, patience=50)

Epoch  30  loss: 1.1666  val AUC: 0.5002
Early stopping at epoch 51  |  Best Val AUC: 0.5002


In [101]:
def evaluate_dl(model, val_loader, device):
    model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for cb, num_b, yb in val_loader:
            logits = model(cb.to(device), num_b.to(device))
            pred = torch.softmax(logits, dim=1).cpu()
            vp.append(pred)
            vt.append(yb)

    vp_np = torch.cat(vp).numpy()
    vt_np = torch.cat(vt).numpy()

    val_auc = roc_auc_score(vt_np, vp_np, multi_class='ovr')  # or 'ovo'
    report = classification_report(vt_np, vp_np.argmax(axis=1))
    cm = confusion_matrix(vt_np, vp_np.argmax(axis=1))
    print(f"Validation AUC: {val_auc:.4f}")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)
    return val_auc

In [102]:
lstm_eval_result = evaluate_dl(lstm_clf, val_loader, device)    

Validation AUC: 0.4690
Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.02      0.03       901
           1       0.52      0.83      0.64      1621
           2       0.13      0.12      0.12       603

    accuracy                           0.46      3125
   macro avg       0.34      0.32      0.27      3125
weighted avg       0.40      0.46      0.37      3125

Confusion Matrix:
[[  14  694  193]
 [  21 1342  258]
 [   4  529   70]]


In [100]:
lstm_eval_result

0.46900493080983735